# 02 — SFT Baseline Training

**Project**: LLM Fine-Tuning with DPO  
**Author**: Pranay Hedau  
**Runs on**: Kaggle T4 GPU (15GB VRAM)  

### What this notebook does
Trains a Supervised Fine-Tuning (SFT) baseline on the chosen responses from UltraFeedback.
This SFT model serves two purposes:
1. **Reference model** for DPO in notebook 03 — DPO measures how far the policy drifts from this
2. **Comparison baseline** — we measure win rate of DPO vs SFT to prove DPO adds value

### Why SFT before DPO?
The raw Llama 3.2 3B base model predicts tokens but doesn't follow instructions well.
SFT is a technique that teaches it the instruction-following format first. DPO then refines *which style*
of instruction-following is preferred. Skipping SFT and going straight to DPO produces
weaker alignment because the reference model has no instruction baseline.

### Kaggle Setup Checklist
Before running this notebook on Kaggle:
- [ ] GPU enabled: Settings → Accelerator → GPU T4 x1
- [ ] Internet enabled: Settings → Internet → On
- [ ] HF_TOKEN added: Add-ons → Secrets → Add `HF_TOKEN`
- [ ] Upload `ultrafeedback_10k.jsonl` as a Kaggle Dataset and attach it

## 1. Environment Check

In [ ]:
import os
import torch

# ── Verify GPU ────────────────────────────────────────────────
print('=== ENVIRONMENT CHECK ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM      : {total_vram:.1f} GB')
else:
    print('WARNING: No GPU detected. This notebook must run on Kaggle T4.')
    print('Go to Settings → Accelerator → GPU T4 x1')

# ── Verify HF token ───────────────────────────────────────────
# On Kaggle: Add-ons → Secrets → HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('HF_TOKEN        : Found via Kaggle Secrets')
except Exception:
    # Fallback for local testing
    from dotenv import load_dotenv
    load_dotenv('../.env')
    HF_TOKEN = os.getenv('HF_TOKEN')
    print('HF_TOKEN        : Loaded from .env')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found. Add it via Kaggle Secrets.')

print()
print('All checks passed. Ready to train.')

## 2. Install Dependencies

Kaggle has most libraries pre-installed but we need specific versions of TRL and PEFT.

In [ ]:
# Run this cell once — takes ~2 minutes on Kaggle
import subprocess
packages = [
    'trl>=0.8.6',
    'peft>=0.10.0',
    'bitsandbytes>=0.43.0',
    'accelerate>=0.29.0',
    'transformers>=4.40.0',
    'datasets>=2.19.0',
]
subprocess.run(['pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

## 3. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

print('Imports complete.')

## 4. Configuration

All hyperparameters in one place. This makes ablation studies easy — 
change one value here and rerun, rather than hunting through the notebook.

In [ ]:
# ── Model ─────────────────────────────────────────────────────
MODEL_ID     = 'meta-llama/Llama-3.2-3B-Instruct'
MAX_SEQ_LEN  = 1024      # max token length — covers 95%+ of UltraFeedback examples

# ── QLoRA ─────────────────────────────────────────────────────
# Why rank=64? Higher rank = more expressiveness but more memory.
# rank=64 is the sweet spot for 3B models on a T4 (15GB VRAM).
LORA_RANK    = 64
LORA_ALPHA   = 16        # scaling factor = alpha/rank. Lower = more conservative updates.
LORA_DROPOUT = 0.05      # small dropout for regularization

# ── Training ──────────────────────────────────────────────────
OUTPUT_DIR   = '/kaggle/working/sft_model'
NUM_EPOCHS   = 1         # 1 epoch is standard for SFT on 10K examples
BATCH_SIZE   = 4         # per device batch size — T4 fits 4 comfortably
GRAD_ACCUM   = 4         # effective batch = 4 * 4 = 16
LR           = 2e-4      # standard for QLoRA fine-tuning
WARMUP_RATIO = 0.03      # 3% warmup steps — prevents loss spike at start
SAVE_STEPS   = 100
LOGGING_STEPS = 25

print('Configuration set:')
print(f'  Model         : {MODEL_ID}')
print(f'  LoRA rank     : {LORA_RANK}')
print(f'  Max seq len   : {MAX_SEQ_LEN}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Learning rate : {LR}')

## 5. Load & Prepare Dataset

SFT trains only on **chosen responses** — we are teaching the model 
to produce good responses, not yet teaching it *preferences*. 
That comes in notebook 03 with DPO.

In [ ]:
def extract_assistant_text(field):
    """Extract assistant response text from chosen/rejected field."""
    if isinstance(field, list):
        for turn in field:
            if turn.get('role') == 'assistant':
                return turn.get('content', '')
        return field[-1].get('content', '') if field else ''
    return str(field)

# Load our curated 10K subset
# On Kaggle: attach your uploaded dataset and update the path below
DATA_PATH = '/kaggle/input/ultrafeedback-10k/ultrafeedback_10k.jsonl'

# Fallback for local testing
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../data/ultrafeedback_10k.jsonl'

raw_dataset = load_dataset('json', data_files=DATA_PATH, split='train')
print(f'Loaded {len(raw_dataset)} examples')

# SFT format: apply the Llama chat template to each example
# Format: <|begin_of_text|><|user|>\nprompt<|end|><|assistant|>\nchosen_response<|end|>
def format_sft(example):
    """
    Format each example as a complete conversation string.
    SFTTrainer expects a 'text' field containing the full formatted prompt+response.
    """
    prompt   = example['prompt']
    response = extract_assistant_text(example['chosen'])
    
    # Llama 3.2 Instruct chat template format
    text = (
        f'<|begin_of_text|>'
        f'<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>'
        f'<|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>'
    )
    return {'text': text}

sft_dataset = raw_dataset.map(format_sft, remove_columns=raw_dataset.column_names)

# Split 90/10 train/val
split = sft_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split['train']
val_data   = split['test']

print(f'Train examples : {len(train_data)}')
print(f'Val examples   : {len(val_data)}')
print()
print('Sample formatted text (first 300 chars):')
print(train_data[0]['text'][:300])

## 6. Load Model with QLoRA

**What QLoRA does in 3 steps:**
1. Load Llama 3.2 3B in 4-bit precision (NF4) — reduces memory from ~12GB to ~3GB
2. Add small trainable LoRA adapter matrices to attention layers
3. Only the adapter weights (~1% of params) are updated during training

The frozen 4-bit base model acts as a compressed knowledge store.
The adapters learn the task-specific adjustments.

In [ ]:
# ── Step 1: Login to HuggingFace ──────────────────────────────
login(token=HF_TOKEN)
print('HuggingFace login successful.')

# ── Step 2: 4-bit quantization config ─────────────────────────
# NF4 (NormalFloat4) is better than INT4 for LLM weights
# because LLM weights follow a normal distribution
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16 for stability
    bnb_4bit_use_double_quant=True,         # quantize the quantization constants too
)

# ── Step 3: Load tokenizer ────────────────────────────────────
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token   # Llama has no pad token by default
tokenizer.padding_side = 'right'            # pad on right for causal LM
print(f'Tokenizer vocab size: {tokenizer.vocab_size:,}')

# ── Step 4: Load model in 4-bit ───────────────────────────────
print('Loading Llama 3.2 3B in 4-bit... (takes ~2-3 minutes)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',      # automatically puts layers on GPU
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False              # disable KV cache during training
model.config.pretraining_tp = 1

# Print memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f'GPU memory after model load: {allocated:.2f} GB')

print('Model loaded successfully.')

In [ ]:
# ── Step 5: Attach LoRA adapters ──────────────────────────────
# We attach adapters to query and value projection layers
# Why q_proj and v_proj? These control what the model attends to
# and how it combines information — the most impactful layers for
# instruction-following behavior.
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Show trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,} ({trainable/total*100:.2f}% of total)')
print(f'Total params     : {total:,}')
print()
print('This is why QLoRA is efficient — we train <2% of parameters.')

## 7. Train SFT Model

**What the loss means here:**  
SFT uses standard cross-entropy loss — the model learns to predict the next token 
in the chosen response given the prompt. Lower loss = better next-token prediction.

**Expected training time**: ~60-90 minutes on Kaggle T4 for 9K examples, 1 epoch.

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',     # cosine decay — standard for LLM fine-tuning
    bf16=True,                      # use bfloat16 for training stability on T4
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    save_total_limit=2,             # keep only last 2 checkpoints to save disk
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',               # disable wandb — not needed for this project
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,                  # don't pack sequences — cleaner for analysis
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
)

print('SFTTrainer configured.')
print(f'Training steps  : {len(trainer.get_train_dataloader()) * NUM_EPOCHS}')
print(f'Eval steps      : every {SAVE_STEPS} steps')
print()
print('Starting training...')
print('Watch for: train/loss decreasing, eval/loss not increasing (no overfitting)')

In [ ]:
# ── Run Training ──────────────────────────────────────────────
# Expected: ~60-90 minutes on Kaggle T4
train_result = trainer.train()

print()
print('=== TRAINING COMPLETE ===')
print(f'Final train loss : {train_result.training_loss:.4f}')
print(f'Total steps      : {train_result.global_step}')
print(f'Runtime          : {train_result.metrics["train_runtime"]:.0f}s')

## 8. Save Model & Verify

In [ ]:
# Save the LoRA adapter weights (not the full model — just the adapters)
# The full model = base Llama 3.2 3B (download from HF) + these adapters
SFT_ADAPTER_DIR = '/kaggle/working/sft_adapter'
trainer.model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)

print(f'SFT adapter saved to: {SFT_ADAPTER_DIR}')

# Show saved files
import os
files = os.listdir(SFT_ADAPTER_DIR)
print(f'Saved files: {files}')

# Check adapter size (much smaller than full model)
total_size = sum(
    os.path.getsize(os.path.join(SFT_ADAPTER_DIR, f))
    for f in files
) / 1e6
print(f'Adapter size: {total_size:.1f} MB  (vs ~6GB for full model)')

## 9. Quick Inference Test

Before saving, sanity check that the SFT model produces reasonable output.
We compare base model vs SFT on the same prompt.

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=200):
    """Generate a response from the model given a prompt."""
    formatted = (
        f'<|begin_of_text|>'
        f'<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>'
        f'<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_prompts = [
    'Explain the difference between supervised and unsupervised learning in simple terms.',
    'Write a Python function that checks if a string is a palindrome.',
    'What are three practical tips for improving sleep quality?',
]

print('=== SFT MODEL INFERENCE TEST ===')
print()
for i, prompt in enumerate(test_prompts, 1):
    response = generate_response(trainer.model, tokenizer, prompt)
    print(f'Prompt {i}: {prompt}')
    print(f'Response: {response[:300]}')
    print('-' * 60)

## 10. Training Summary

In [ ]:
print('=' * 65)
print(f'{"SFT TRAINING SUMMARY":^65}')
print('=' * 65)
print(f'  Model         : {MODEL_ID}')
print(f'  Dataset       : UltraFeedback 10K (chosen responses only)')
print(f'  LoRA rank     : {LORA_RANK}')
print(f'  Final loss    : {train_result.training_loss:.4f}')
print(f'  Adapter saved : {SFT_ADAPTER_DIR}')
print()
print('  NEXT STEPS:')
print('  1. Download sft_adapter/ folder from Kaggle output')
print('  2. Re-upload as a new Kaggle dataset named sft-adapter')
print('  3. Attach it in notebook 03_dpo_training.ipynb')
print('     (DPO uses this as the reference model)')
print('=' * 65)